# 🏭 Visualisation Process Industriel — Graphviz (style Karpathy)

Utilise `graphviz` avec des **labels HTML** pour afficher OEE, CMJ, capacité, site…  
Chaque nœud est un mini-tableau — rendu SVG vectoriel, layout hiérarchique `dot`.

| Fichier | Contenu |
|---|---|
| `sites.json` | Sites industriels |
| `matieres_premieres.json` | MP, stocks, seuils |
| `unites_production.json` | UP, OEE, CMJ, flux |

## 1. Installation & imports

In [5]:
# !pip install graphviz pyvis pandas
# Sur Linux/Mac : sudo apt install graphviz  |  brew install graphviz
# Sur Windows  : https://graphviz.org/download/

import json
import pandas as pd
from pathlib import Path
from graphviz import Digraph
from IPython.display import display, HTML, SVG

print('✅ OK')

✅ OK


## 2. Chargement des données JSON

In [6]:
BASE = Path('.')

def jload(f):
    with open(BASE / f, encoding='utf-8') as fh:
        return json.load(fh)

sites = jload('sites.json')['sites']
mps   = jload('matieres_premieres.json')['matieres_premieres']
ups   = jload('unites_production.json')['unites_production']

# Index rapides
site_map = {s['code']: s for s in sites}
mp_map   = {m['code']: m for m in mps}
up_map   = {u['code']: u for u in ups}

print(f'{len(sites)} sites  |  {len(mps)} MP  |  {len(ups)} UP')

4 sites  |  5 MP  |  7 UP


## 3. 🎨 Helpers : couleurs et labels HTML
*(C'est ici qu'on définit ce qui s'affiche dans chaque nœud — modifiable librement)*

In [7]:
# ── Palette ───────────────────────────────────────────────────────────────────
PALETTE = {
    'mp_header'       : ('#C0392B', '#FADBD8'),   # rouge
    'additif'         : ('#C0392B', '#FADBD8'),
    'consommable'     : ('#C0392B', '#FADBD8'),
    'emballage'       : ('#7D3C98', '#E8DAEF'),   # violet
    'intermediaire'   : ('#D35400', '#FDEBD0'),   # orange
    'fini'            : ('#1A5276', '#D6EAF8'),   # bleu foncé
    'site_banner'     : ('#2C3E50', '#ECF0F1'),
    'oee_green'       : '#1E8449',
    'oee_orange'      : '#D35400',
    'oee_red'         : '#C0392B',
}

def oee_color(val):
    if val >= 85: return PALETTE['oee_green']
    if val >= 75: return PALETTE['oee_orange']
    return PALETTE['oee_red']

def oee_bar(val, width=80):
    """Mini barre de progression HTML pour l'OEE."""
    fill  = int(val / 100 * width)
    color = oee_color(val)
    return (
        f'<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0">'
        f'<TR>'
        f'<TD BGCOLOR="{color}" WIDTH="{fill}" HEIGHT="6"></TD>'
        f'<TD BGCOLOR="#D5D8DC" WIDTH="{width-fill}" HEIGHT="6"></TD>'
        f'</TR></TABLE>'
    )

def pct_stock(mp):
    try:
        return round(mp['volume_stock'] / mp['stock_max'] * 100)
    except Exception:
        return 0

def stock_color(mp):
    p = pct_stock(mp)
    if mp['volume_stock'] <= mp['stock_min']: return PALETTE['oee_red']
    if p < 30: return PALETTE['oee_orange']
    return PALETTE['oee_green']

# ── Label HTML — Matière première ─────────────────────────────────────────────
def label_mp(mp):
    typ  = mp['type']
    hbg, cbg = PALETTE.get(typ, PALETTE['mp_header'])
    pct  = pct_stock(mp)
    sc   = stock_color(mp)
    fill = int(pct * 0.8)
    return (
        f'<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4" BGCOLOR="{cbg}">'
        f'<TR><TD COLSPAN="2" BGCOLOR="{hbg}" ALIGN="CENTER">'
        f'<FONT COLOR="white" POINT-SIZE="10"><B>{mp["nom"]}</B></FONT></TD></TR>'
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Code</FONT></TD>'
        f'    <TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{mp["code"]}</FONT></TD></TR>'
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Type</FONT></TD>'
        f'    <TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{typ}</FONT></TD></TR>'
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Stock</FONT></TD>'
        f'    <TD ALIGN="RIGHT"><FONT POINT-SIZE="8" COLOR="{sc}">'
        f'<B>{mp["volume_stock"]:,} {mp["unite_volume"]}</B></FONT></TD></TR>'
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">'
        f'<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0">'
        f'<TR><TD BGCOLOR="{sc}" WIDTH="{fill}" HEIGHT="5"></TD>'
        f'    <TD BGCOLOR="#D5D8DC" WIDTH="{80-fill}" HEIGHT="5"></TD></TR>'
        f'</TABLE></TD></TR>'
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">'
        f'<FONT POINT-SIZE="7" COLOR="{sc}">Stock {pct}% du max</FONT>'
        f'</TD></TR>'
        f'</TABLE>>'
    )

# ── Label HTML — Unité de production ─────────────────────────────────────────
def label_up(up):
    typ  = up['type_produit']
    key  = 'intermediaire' if 'inter' in typ else 'fini'
    hbg, cbg = PALETTE[key]
    oc   = oee_color(up['oee'])
    site = site_map.get(up['site_code'], {}).get('nom', up['site_code'])
    taux = round(up['cmj'] / up['capacite_max_j'] * 100)
    fill = int(taux * 0.8)
    efill = int(up['oee'] / 100 * 80)
    border = '3' if up['type_ligne'] == 'principale' else '1'
    return (
        f'<<TABLE BORDER="{border}" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4" BGCOLOR="{cbg}">'
        # En-tête
        f'<TR><TD COLSPAN="2" BGCOLOR="{hbg}" ALIGN="CENTER">'
        f'<FONT COLOR="white" POINT-SIZE="11"><B>{up["nom"]}</B></FONT></TD></TR>'
        # Produit
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">'
        f'<FONT POINT-SIZE="9" COLOR="#2C3E50">📦 {up["produit"]}</FONT></TD></TR>'
        # Site
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">🏭 Site</FONT></TD>'
        f'    <TD ALIGN="RIGHT"><FONT POINT-SIZE="8"><B>{site}</B></FONT></TD></TR>'
        # Code & type ligne
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Code</FONT></TD>'
        f'    <TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{up["code"]} '
        f'({up["type_ligne"]})</FONT></TD></TR>'
        # Séparateur
        f'<TR><TD COLSPAN="2" BGCOLOR="{hbg}" HEIGHT="1"></TD></TR>'
        # OEE
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="9"><B>OEE</B></FONT></TD>'
        f'    <TD ALIGN="RIGHT"><FONT POINT-SIZE="9" COLOR="{oc}">'
        f'<B>{up["oee"]}%</B></FONT></TD></TR>'
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">'
        f'<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0">'
        f'<TR><TD BGCOLOR="{oc}" WIDTH="{efill}" HEIGHT="6"></TD>'
        f'    <TD BGCOLOR="#D5D8DC" WIDTH="{80-efill}" HEIGHT="6"></TD></TR>'
        f'</TABLE></TD></TR>'
        # CMJ
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">CMJ</FONT></TD>'
        f'    <TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{up["cmj"]:,} {up["unite_capacite"]}</FONT></TD></TR>'
        # Capacité max
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Cap. max/j</FONT></TD>'
        f'    <TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{up["capacite_max_j"]:,}</FONT></TD></TR>'
        # Taux de charge
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Taux charge</FONT></TD>'
        f'    <TD ALIGN="RIGHT"><FONT POINT-SIZE="8"><B>{taux}%</B></FONT></TD></TR>'
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">'
        f'<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0">'
        f'<TR><TD BGCOLOR="#2980B9" WIDTH="{fill}" HEIGHT="5"></TD>'
        f'    <TD BGCOLOR="#D5D8DC" WIDTH="{80-fill}" HEIGHT="5"></TD></TR>'
        f'</TABLE></TD></TR>'
        f'</TABLE>>'
    )

print('✅ Helpers OK')

✅ Helpers OK


## 4. 🕸️ Construction du graphe Graphviz

In [8]:
def build_graph(
    show_mp      = True,
    site_filter  = None,      # None = tous les sites | 'SITE-A' = un seul
    rankdir      = 'LR',      # 'LR' gauche→droite  |  'TB' haut→bas
    show_site_clusters = True,
):
    """
    Construit et retourne un objet graphviz.Digraph.
    
    Paramètres
    ----------
    show_mp             : inclure les nœuds matières premières
    site_filter         : filtrer sur un seul site (ou None = tous)
    rankdir             : direction du layout
    show_site_clusters  : regrouper les UP par cluster/site
    """
    dot = Digraph(
        name='process_industriel',
        format='svg',
        graph_attr={
            'rankdir'   : rankdir,
            'splines'   : 'ortho',       # arêtes orthogonales (style industriel)
            'nodesep'   : '0.6',
            'ranksep'   : '1.0',
            'bgcolor'   : '#F8F9FA',
            'fontname'  : 'Helvetica',
            'label'     : '🏭  PROCESS INDUSTRIEL MULTI-SITES',
            'labelloc'  : 't',
            'fontsize'  : '16',
            'fontcolor' : '#2C3E50',
        },
        node_attr={
            'shape'    : 'plaintext',  # nécessaire pour les labels HTML tabulés
            'fontname' : 'Helvetica',
        },
        edge_attr={
            'fontname' : 'Helvetica',
            'fontsize' : '8',
        },
    )

    # Filtre UP
    ups_filtered = [
        u for u in ups
        if site_filter is None or u['site_code'] == site_filter
    ]
    up_codes_filtered = {u['code'] for u in ups_filtered}

    # ── Clusters par site ─────────────────────────────────────────────────────
    if show_site_clusters:
        active_sites = {u['site_code'] for u in ups_filtered}
        for s in sites:
            if s['code'] not in active_sites:
                continue
            with dot.subgraph(name=f'cluster_{s["code"]}') as sg:
                sg.attr(
                    label     = f'{s["nom"]}  •  {s["localisation"]}',
                    style     = 'rounded,filled',
                    fillcolor = s.get('couleur','#ECF0F1') + '22',  # alpha
                    color     = s.get('couleur','#2C3E50'),
                    fontsize  = '12',
                    fontcolor = s.get('couleur','#2C3E50'),
                    penwidth  = '2',
                )
                for u in ups_filtered:
                    if u['site_code'] == s['code']:
                        sg.node(u['code'], label=label_up(u))
    else:
        for u in ups_filtered:
            dot.node(u['code'], label=label_up(u))

    # ── Nœuds Matières premières ──────────────────────────────────────────────
    if show_mp:
        for mp in mps:
            if site_filter and mp['site_code'] != site_filter:
                continue
            dot.node(mp['code'], label=label_mp(mp))

    # ── Arêtes MP → UP ────────────────────────────────────────────────────────
    if show_mp:
        for u in ups_filtered:
            for src in u.get('matieres_premieres_aval', []):
                if src.startswith('MP-') and src in mp_map:
                    mp = mp_map[src]
                    if site_filter and mp['site_code'] != site_filter:
                        continue
                    dot.edge(
                        src, u['code'],
                        color     = '#C0392B',
                        style     = 'dashed',
                        arrowsize = '0.7',
                        penwidth  = '1.2',
                        label     = mp_map[src]['nom'][:15],
                    )

    # ── Arêtes UP → UP ────────────────────────────────────────────────────────
    for u in ups_filtered:
        for nxt in u.get('produits_suivants', []):
            if nxt not in up_codes_filtered and site_filter:
                continue  # skip flux hors périmètre filtré
            if nxt not in {x['code'] for x in ups}:
                continue
            is_main = u['type_ligne'] == 'principale'
            dot.edge(
                u['code'], nxt,
                color     = '#2C3E50' if is_main else '#7F8C8D',
                penwidth  = '2.5' if is_main else '1.2',
                style     = 'solid' if is_main else 'dashed',
                arrowsize = '1.0' if is_main else '0.7',
                label     = u['produit'][:20] if is_main else '',
                fontsize  = '8',
                fontcolor = '#2C3E50',
            )

    return dot

print('✅ Fonction build_graph() prête')

✅ Fonction build_graph() prête


## 5. 📊 Vue complète — tous sites, toutes MP

In [9]:
g = build_graph(show_mp=True, rankdir='LR', show_site_clusters=True)
display(SVG(g.pipe(format='svg')))

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

## 6. ⚙️ Vue flux principaux — sans matières premières

In [ ]:
g2 = build_graph(show_mp=False, rankdir='TB', show_site_clusters=True)
display(SVG(g2.pipe(format='svg')))

## 7. 🔍 Vue filtrée par site

In [ ]:
# Afficher les sites disponibles
for s in sites:
    print(f"  {s['code']}  →  {s['nom']} ({s['localisation']})")

# ← Changer le code ici
g3 = build_graph(show_mp=True, site_filter='SITE-A', rankdir='TB')
display(SVG(g3.pipe(format='svg')))

## 8. 💾 Export SVG / PDF / PNG

In [ ]:
g_export = build_graph(show_mp=True, rankdir='LR', show_site_clusters=True)

# SVG (vectoriel — recommandé)
g_export.format = 'svg'
g_export.render('process_industriel', cleanup=True)
print('✅ Exporté : process_industriel.svg')

# PDF
g_export.format = 'pdf'
g_export.render('process_industriel_print', cleanup=True)
print('✅ Exporté : process_industriel_print.pdf')

# PNG haute résolution (nécessite Graphviz installé)
import subprocess
try:
    subprocess.run(
        ['dot', '-Tpng', '-Gdpi=200', 'process_industriel.svg',
         '-o', 'process_industriel.png'],
        check=True
    )
    print('✅ Exporté : process_industriel.png (200 dpi)')
except Exception as e:
    print(f'⚠️  PNG skippé : {e}')

## 9. 🌐 Bonus — PyVis (vue interactive navigable)
*Alternative graphviz si vous voulez zoomer/déplacer les nœuds à la souris*

In [ ]:
from pyvis.network import Network
import networkx as nx

net = Network(height='700px', width='100%', directed=True,
              bgcolor='#F8F9FA', font_color='#2C3E50')
net.set_options("""
{
  "physics": {"enabled": true, "solver": "hierarchicalRepulsion"},
  "layout": {"hierarchical": {"enabled": true, "direction": "LR", "sortMethod": "directed"}}
}
""")

for mp in mps:
    pct = pct_stock(mp)
    sc  = stock_color(mp)
    net.add_node(
        mp['code'],
        label = f"{mp['nom']}\n{mp['volume_stock']:,} {mp['unite_volume']}",
        title = f"<b>{mp['nom']}</b><br>Stock: {mp['volume_stock']:,} {mp['unite_volume']}<br>{pct}% du max",
        color = {'background': sc, 'border': '#2C3E50'},
        shape = 'diamond',
        size  = 20,
        font  = {'color': 'white', 'size': 11},
    )

for u in ups:
    oc    = oee_color(u['oee'])
    taux  = round(u['cmj'] / u['capacite_max_j'] * 100)
    shape = 'box' if u['type_produit'] == 'produit_fini' else 'ellipse'
    net.add_node(
        u['code'],
        label = f"{u['nom']}\nOEE {u['oee']}% | CMJ {u['cmj']:,}",
        title = (f"<b>{u['nom']}</b><br>"
                 f"Produit : {u['produit']}<br>"
                 f"OEE : {u['oee']}%<br>"
                 f"CMJ : {u['cmj']:,} {u['unite_capacite']}<br>"
                 f"Cap. max : {u['capacite_max_j']:,}<br>"
                 f"Taux charge : {taux}%<br>"
                 f"Site : {site_map[u['site_code']]['nom']}"),
        color = {'background': oc, 'border': '#2C3E50'},
        shape = shape,
        size  = 35 if u['type_ligne'] == 'principale' else 25,
        font  = {'color': 'white', 'size': 12},
    )

for u in ups:
    for src in u.get('matieres_premieres_aval', []):
        if src.startswith('MP-'):
            net.add_edge(src, u['code'], color='#C0392B', dashes=True, width=1)
    for nxt in u.get('produits_suivants', []):
        is_main = u['type_ligne'] == 'principale'
        net.add_edge(u['code'], nxt,
                     color   = '#2C3E50' if is_main else '#7F8C8D',
                     width   = 3 if is_main else 1.5,
                     dashes  = not is_main,
                     label   = u['produit'][:18] if is_main else '',
                    )

net.save_graph('process_industriel_interactive.html')
print('✅ Fichier HTML interactif généré : process_industriel_interactive.html')
print('   → Ouvrir dans un navigateur pour naviguer, zoomer, déplacer les nœuds')

---
## 💡 Référence rapide

| Action | Code |
|---|---|
| Vue complète | `build_graph()` |
| Sans MP | `build_graph(show_mp=False)` |
| Haut→bas | `build_graph(rankdir='TB')` |
| Un site | `build_graph(site_filter='SITE-B')` |
| Export SVG | `g.render('nom', cleanup=True)` |
| Vue interactive | Section 9 (PyVis HTML) |
| Ajouter un nœud | Editer le JSON correspondant |
| Modifier un label | Fonctions `label_mp()` / `label_up()` |
